The results are cystallized into the [`create_embbeddings.py` script](../lib/create_embeddings.py).

In [7]:
import polars as pl
from pprint import pprint
from sentence_transformers import SentenceTransformer

In [8]:
data = pl.scan_ndjson("../data/raw/arxiv-metadata-oai-snapshot.json")

In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
filtered_data = (
    data
    .head(1000)
    .filter(pl.col("abstract").str.len_chars() > 150)
    .select(
        pl.col("id"),
        (pl.lit("Title: ") + pl.col("title") + pl.lit(" | Abstract: ") + pl.col("abstract")).alias("content")
    )
    .with_columns(
        pl.col("content").map_batches(
            lambda s: pl.Series(
                embedder.encode(s.to_list(), batch_size=128, precision="float32").tolist(),
                dtype=pl.List(pl.Float32)
                ),
            return_dtype=pl.List(pl.Float32)
        ).alias("embedding")
    )
)

In [11]:
filtered_data.head()

In [12]:
embedded_filtered_data = filtered_data.collect()

In [13]:
pprint(embedded_filtered_data[0,1])

('Title: Calculation of prompt diphoton production cross sections at Tevatron '
 'and\n'
 '  LHC energies | Abstract:   A fully differential calculation in '
 'perturbative quantum chromodynamics is\n'
 'presented for the production of massive photon pairs at hadron colliders. '
 'All\n'
 'next-to-leading order perturbative contributions from quark-antiquark,\n'
 'gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\n'
 'all-orders resummation of initial-state gluon radiation valid at\n'
 'next-to-next-to-leading logarithmic accuracy. The region of phase space is\n'
 'specified in which the calculation is most reliable. Good agreement is\n'
 'demonstrated with data from the Fermilab Tevatron, and predictions are made '
 'for\n'
 'more detailed tests with CDF and DO data. Predictions are shown for\n'
 'distributions of diphoton pairs produced at the energy of the Large Hadron\n'
 'Collider (LHC). Distributions of the diphoton pairs from the decay of a '
 'Higgs\n'
 '

In [14]:
len(embedded_filtered_data[0,2])

384

In [15]:
filtered_data.sink_parquet("../data/processed/arxiv_embeddings.parquet")

In [16]:
reloaded_data = pl.read_parquet("../data/processed/arxiv_embeddings.parquet")

In [17]:
reloaded_data

id,content,embedding
str,str,list[f32]
"""0704.0001""","""Title: Calculation of prompt d…","[-0.158678, -0.010544, … 0.049717]"
"""0704.0002""","""Title: Sparsity-certifying Gra…","[0.011342, 0.034085, … 0.021261]"
"""0704.0003""","""Title: The evolution of the Ea…","[0.031139, -0.031973, … 0.004312]"
"""0704.0004""","""Title: A determinant of Stirli…","[-0.068851, -0.018202, … 0.000149]"
"""0704.0005""","""Title: From dyadic $\Lambda_{\…","[0.035553, 0.003334, … -0.026612]"
…,…,…
"""0704.0996""","""Title: Brane World Black Rings…","[-0.012916, -0.040612, … -0.027583]"
"""0704.0997""","""Title: Stable algebras of enti…","[-0.053272, 0.056141, … 0.053394]"
"""0704.0998""","""Title: Test vectors for trilin…","[-0.113309, -0.035348, … 0.017367]"
